# 🛡️ Sovereign-Vision — Phase 2: YOLOv8s Training

**Pipeline:**
1. Verify GPU
2. Clone repo from GitHub
3. Mount Google Drive (dataset)
4. Install dependencies
5. Link dataset into repo structure
6. Run desert augmentation on training set
7. Train YOLOv8s
8. Evaluate on test set
9. Run inference on sample images
10. Save best weights to Google Drive

**Before running:** Make sure `dataset/` is uploaded to Google Drive at:
`My Drive/sovereign-vision/dataset/`

## 0. Verify GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")

## 1. Config & Clone Repository

In [ ]:
import os

# ── UPDATE THESE ──────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/YOUR_USERNAME/sovereign-vision.git"
DRIVE_DATASET = "/content/drive/MyDrive/sovereign-vision/dataset"
# ─────────────────────────────────────────────────────────────────

PROJECT_DIR = "/content/sovereign-vision"
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 16
MODEL       = "yolov8s.pt"

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.exists(DRIVE_DATASET), f"Dataset not found at {DRIVE_DATASET}"

for split in ["train", "val", "test"]:
    img_dir = os.path.join(DRIVE_DATASET, "images", split)
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"  {split:<6}: {count} images")

print("\n✓ Google Drive mounted and dataset verified")

## 3. Install Dependencies

In [ ]:
!pip install ultralytics albumentations --quiet

from ultralytics import YOLO
import albumentations as A
import cv2
print("✓ ultralytics installed")
print("✓ albumentations installed")

## 4. Link Dataset + Write data.yaml

In [ ]:
import yaml
from pathlib import Path

local_dataset = Path(PROJECT_DIR) / "dataset"
if not local_dataset.exists():
    os.symlink(DRIVE_DATASET, str(local_dataset))
    print(f"✓ Symlinked {DRIVE_DATASET} → {local_dataset}")
else:
    print(f"✓ Dataset already linked at {local_dataset}")

yaml_content = {
    "path":  DRIVE_DATASET,
    "train": os.path.join(DRIVE_DATASET, "images", "train"),
    "val":   os.path.join(DRIVE_DATASET, "images", "val"),
    "test":  os.path.join(DRIVE_DATASET, "images", "test"),
    "nc":    4,
    "names": {0: "vehicle", 1: "truck", 2: "personnel", 3: "equipment"}
}

yaml_path = Path("/content/data_colab.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"✓ data_colab.yaml written")
!cat {yaml_path}

## 5. Desert Augmentation on Training Set

> **Optional** — skip to cell 6 to train on raw data first as a baseline.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from pipeline.augment import augment_dataset, DEFAULT_MODES
from pathlib import Path
import shutil

AUG_DIR      = Path("/content/augmented")
TRAIN_IMAGES = Path(DRIVE_DATASET) / "images" / "train"
TRAIN_LABELS = Path(DRIVE_DATASET) / "labels" / "train"

print("Running desert augmentation...")
print(f"  Modes  : {DEFAULT_MODES}")
print(f"  Copies : 3 per image\n")

stats = augment_dataset(
    images_dir=TRAIN_IMAGES,
    labels_dir=TRAIN_LABELS,
    output_images_dir=AUG_DIR / "images" / "train",
    output_labels_dir=AUG_DIR / "labels" / "train",
    copies=3,
    modes=DEFAULT_MODES,
    thermal_copy=True,
)

for split in ["val", "test"]:
    for kind in ["images", "labels"]:
        src = Path(DRIVE_DATASET) / kind / split
        dst = AUG_DIR / kind / split
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    print(f"✓ Copied {split}")

total = stats['source'] + stats['augmented'] + stats['thermal']
print(f"\n✓ Augmented train set: {total} images")

In [ ]:
AUG_DIR.mkdir(parents=True, exist_ok=True)

aug_yaml_content = {
    "path":  str(AUG_DIR),
    "train": str(AUG_DIR / "images" / "train"),
    "val":   str(AUG_DIR / "images" / "val"),
    "test":  str(AUG_DIR / "images" / "test"),
    "nc":    4,
    "names": {0: "vehicle", 1: "truck", 2: "personnel", 3: "equipment"}
}

aug_yaml_path = Path("/content/data_augmented.yaml")
with open(aug_yaml_path, "w") as f:
    yaml.dump(aug_yaml_content, f, default_flow_style=False)

aug_count = len(list((AUG_DIR / "images" / "train").glob("*")))
print(f"✓ Augmented yaml written")
print(f"✓ Augmented train images: {aug_count}")

## 6. Train YOLOv8s

In [ ]:
from ultralytics import YOLO

DATA_YAML = str(aug_yaml_path)   # augmented
# DATA_YAML = str(yaml_path)     # raw baseline

model = YOLO(MODEL)

print(f"Model    : {MODEL}")
print(f"Data     : {DATA_YAML}")
print(f"Epochs   : {EPOCHS}")
print()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name="sovereign-vision-yolov8s",
    project="/content/runs",
    device=0,
    patience=15,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
)

## 7. Evaluate on Test Set

In [ ]:
best_weights = "/content/runs/sovereign-vision-yolov8s/weights/best.pt"
model_eval = YOLO(best_weights)

metrics = model_eval.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    plots=True,
    save_json=True,
)

print("\n── Test Set Results ──────────────────────")
print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")
print("──────────────────────────────────────────")

## 8. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = "/content/runs/sovereign-vision-yolov8s"
plot_files = [
    ("results.png",          "Training Results"),
    ("confusion_matrix.png", "Confusion Matrix"),
    ("PR_curve.png",         "Precision-Recall Curve"),
    ("F1_curve.png",         "F1 Curve"),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, (filename, title) in zip(axes, plot_files):
    path = os.path.join(run_dir, filename)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=13, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{filename} not found",
                ha="center", va="center", transform=ax.transAxes)
    ax.axis("off")

plt.suptitle("Sovereign-Vision — YOLOv8s Training Results", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/training_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: /content/training_summary.png")

## 9. Inference on Sample Test Images

In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

test_images_dir = Path(DRIVE_DATASET) / "images" / "test"
test_images = list(test_images_dir.glob("*.jpg")) + list(test_images_dir.glob("*.png"))
samples = random.sample(test_images, min(8, len(test_images)))

results_inf = model_eval.predict(
    source=samples,
    imgsz=IMG_SIZE,
    conf=0.25,
    iou=0.45,
    device=0,
    save=False,
    verbose=False,
)

CLASS_COLORS = {0:(0,200,255), 1:(0,100,255), 2:(0,255,100), 3:(255,80,80)}
CLASS_NAMES  = {0:"vehicle", 1:"truck", 2:"personnel", 3:"equipment"}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, result in zip(axes, results_inf):
    img = cv2.imread(str(result.path))
    img = cv2.cvtColor(cv2.resize(img, (640, 640)), cv2.COLOR_BGR2RGB)
    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            c = CLASS_COLORS.get(cls_id,(255,255,255))
            cv2.rectangle(img,(x1,y1),(x2,y2),(c[2],c[1],c[0]),2)
            cv2.putText(img,f"{CLASS_NAMES.get(cls_id,cls_id)} {conf:.2f}",
                        (x1,max(y1-5,10)),cv2.FONT_HERSHEY_SIMPLEX,0.4,(c[2],c[1],c[0]),1)
    n_det = len(result.boxes) if result.boxes else 0
    ax.imshow(img)
    ax.set_title(f"{Path(result.path).stem[:15]} — {n_det} det.", fontsize=9)
    ax.axis("off")

plt.suptitle("Sovereign-Vision — YOLOv8s Inference on Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/inference_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: /content/inference_samples.png")

## 10. Save Best Weights to Google Drive

In [ ]:
import shutil
from pathlib import Path

drive_output = Path("/content/drive/MyDrive/sovereign-vision/runs")
drive_output.mkdir(parents=True, exist_ok=True)
run_dir = Path("/content/runs/sovereign-vision-yolov8s")

weights_dst = drive_output / "weights"
if weights_dst.exists(): shutil.rmtree(weights_dst)
shutil.copytree(run_dir / "weights", weights_dst)
print(f"✓ Weights → {weights_dst}")

for f in ["results.png","confusion_matrix.png","PR_curve.png","F1_curve.png"]:
    src = run_dir / f
    if src.exists():
        shutil.copy2(src, drive_output / f)
        print(f"✓ {f} → Drive")

for f in ["/content/training_summary.png","/content/inference_samples.png"]:
    if os.path.exists(f):
        shutil.copy2(f, drive_output / Path(f).name)
        print(f"✓ {Path(f).name} → Drive")

print(f"\n✓ All outputs saved to {drive_output}")

## 11. Final Summary

In [ ]:
print("═" * 52)
print("  SOVEREIGN-VISION — TRAINING COMPLETE")
print("═" * 52)
print(f"  Model         : YOLOv8s")
print(f"  Epochs        : {EPOCHS}")
print(f"  Image size    : {IMG_SIZE}")
print(f"  mAP@0.5       : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print(f"  Best weights  : {weights_dst / 'best.pt'}")
print("═" * 52)